# Feature & Label EDA — SPY daily

Exploratory look at the cached SPY series and the engineered features/labels before the study. The leakage discipline itself lives in `src/market_prediction_study/features.py` and is enforced by `tests/test_features.py`; this notebook is for *intuition*: distributions, persistence, and the weak autocorrelation that motivates the whole study.

**Run it after `python scripts/download_data.py`.**

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve() / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from market_prediction_study.config import StudyConfig
from market_prediction_study.data import load_dataset
from market_prediction_study.features import build_features

## 1. Cached data

In [ ]:
raw = load_dataset(StudyConfig())
print(raw.shape)
raw[["close", "volume"]].tail(3)

## 2. Returns: fat tails, tiny autocorrelation

In [ ]:
feats = build_features(raw)
ret = feats["log_ret_lag_1"]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ret.plot(kind="hist", bins=80, ax=ax[0], density=True, color="tab:blue")
ax[0].set_title("Daily log returns (lag-1 feature): leptokurtic")
ax[1].bar(range(1, 11), [ret.autocorr(l) for l in range(1, 11)], color="tab:orange")
ax[1].set_title("Autocorrelation of daily returns, lags 1-10")
ax[1].axhline(0, color="k", lw=0.8)
fig.tight_layout(); plt.show()

print("max |autocorr| over lags 1-10:", max(abs(ret.autocorr(l)) for l in range(1, 11)).round(4))

## 3. The label base rate is the real benchmark

In [ ]:
up = feats["target_up"]
print(f"up-day base rate: {up.mean():.3f}  (majority baseline accuracy = this number)")
up.resample("YE").mean().plot(kind="bar", figsize=(10, 3), color="tab:green", legend=False)
plt.title("Up-day share by year — persistence varies by regime"); plt.tight_layout(); plt.show()

## Takeaways

- Daily return autocorrelations are tiny (order 0.0x) — any learnable signal is at most a few basis points of edge.
- The up-day base rate (≈0.55) makes "always up" a strong null; models must beat *it*, not zero.
- Regime dependence of the base rate means a single bull-dominated holdout inflates the baseline — one of the documented limitations.